# 第 7 周第 2 天

# 「THE PRICE IS RIGHT」毕业项目

本周——微调一个开源模型！

一个能根据商品描述估算价格的模型。

# 日程安排

第 1 天：QLoRA  
第 2 天：Prompt 数据与基础模型  
第 3 天：训练（上）  
第 4 天：训练（下）  
第 5 天：评估 

## 首先我们需要上传最终数据集




In [ ]:
# 导入：环境变量、Hugging Face 登录、商品 Item、分词器与绘图
# 本课为微调（fine-tuning）做数据准备：先看 token 长度再生成 prompt

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt


In [ ]:
# LITE_MODE=True 时用小数据集快速试跑；False 用完整数据集
# 从 .env 读取 HF_TOKEN 并登录 Hugging Face Hub

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
# 从 Hub 拉取商品数据集（lite / full），合并 train/val/test 方便统计

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# 基座模型：Llama 3.2 3B（后续微调会基于它）
# AutoTokenizer 把文本切成模型能理解的 token

BASE_MODEL = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
# 统计每条商品 summary 的 token 数，为截断阈值 CUTOFF 做准备

token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
# 画直方图：看 summary 长度分布，帮助选择合理的 CUTOFF

plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()

In [ ]:
# CUTOFF：超过该 token 数的样本会被截断；打印会受影响的比例

CUTOFF = 110
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With this CUTOFF, we will truncate {cut:,} items which is {cut/len(items):.1%}")


In [ ]:
# 看一条训练样本的 summary 长什么样

print(train[0].summary)

In [ ]:
# 为 train/val 生成带答案的 prompt（监督学习）；test 不泄露价格
# make_prompts 会按 CUTOFF 截断，并把文本整理成微调格式

for item in tqdm(train+val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)

In [ ]:
# 检查测试集样本：prompt 是输入，completion 是期望输出（价格）

print("PROMPT:")
print(test[0].prompt)
print("COMPLETION:")
print(test[0].completion)


In [ ]:
# 再统计「prompt + completion」总 token，确认不会超出模型上下文

prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
# 可视化 prompt+completion 的 token 分布

plt.figure(figsize=(15, 6))
plt.title(f"Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel('Number of tokens in prompt and the completion')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()

In [ ]:
# 把整理好的 prompts 推送到 Hugging Face Hub，供第 3–4 天 Colab 微调使用

username = "ed-donner"
dataset = f"{username}/items_prompts_lite" if LITE_MODE else f"{username}/items_prompts_full"

Item.push_prompts_to_hub(dataset, train, val, test)

以下是 HuggingFace 上的数据集：

https://huggingface.co/datasets/ed-donner/items_prompts_lite

https://huggingface.co/datasets/ed-donner/items_prompts_full

请在 Google Colab 中查看本 notebook：

https://colab.research.google.com/drive/1wO3lNMrMfprlJZF4X9fSsQ8tYC3SRZbh?usp=sharing